In [ ]:
!pip install --upgrade "plotly>=5.4"

In [1]:
import pandas as pd
import plotly.express as px


csv_paths = [
    "2025_1.csv",
    "2025_0.csv"
]
dfs = []
for f in csv_paths:
    dfs.append(pd.read_csv(
        f,
        sep=";",                # semicolon separator
        quotechar='"',          # handle quoted fields properly
        decimal=",",            # interpret ',' as decimal
        engine="python",        # more forgiving parser
        dtype=str               # read everything as string first to avoid type issues
    ))
df = pd.concat(dfs, ignore_index=True)

In [9]:
df.dtypes

Data księgowania            datetime64[ns]
Data waluty                         object
Nadawca / Odbiorca                  object
Adres nadawcy / odbiorcy            object
Rachunek źródłowy                   object
Rachunek docelowy                   object
Tytułem                             object
Kwota operacji                     float64
Waluta                              object
Numer referencyjny                  object
Typ operacji                        object
Kategoria                           object
Kategoria_2                         object
Miesiąc                          period[M]
Miesiąc_etykieta                    object
dtype: object

In [6]:
category_map = {
    'Zdrowie': 
        ['Lekarstwa','Opieka medyczna'],
    'Transport': 
        ['Paliwo','Transport publiczny','Taxi','Bilety lotnicze'],
    'Odzież i dodatki': 
        ['Dodatki, biżuteria','Ubrania','Usługi (pralnia, krawiec, szewc,...)'],
    'Rachunki': 
        ['Czynsz','Internet, TV, telefon','Ubezpieczenia','Prąd'],
    'Dom': 
        ['Artykuły dekoracyjne','Naprawy i remonty','Wyposażenie','Sprzęt AGD i RTV','Ogród','Sprzątanie'],
    'Rozrywka i wypoczynek': 
        ['Restauracje i kawiarnie','Multimedia','Sport','Podróże','Puby i kluby','Gazety lub czasopisma','Kino i teatr','Hobby','Hotele','Książki'],
    'Wydatki bieżące': 
        ['Artykuły spożywcze','Kosmetyki','Zwierzęta domowe','Uroda, fryzjer, kosmetyczka','Fotografia','Zakupy przez internet','Alkohol','Papierosy'],
    'Dla innych': 
        ['Prezenty, upominki'],
    'Podatki i opłaty': 
        ['Opłaty bankowe'],
    'Obciążenia wewnętrzne': 
        ['Przelew wewnętrzny','Założenie lokaty, zakup funduszy, akcji'],
    'Inne': 
        ['Inne','Wypłata z bankomatu','Spłata kredytu / pożyczki','Wypłata z rachunku'],
    'Bez kategorii': 
        ['Bez kategorii']
}
reverse_map = {v:k for k,vals in category_map.items() for v in vals}
df["Kategoria_2"] = df["Kategoria"].map(reverse_map)  # .fillna("Bez kategorii")

In [7]:
df["Data księgowania"] = pd.to_datetime(df["Data księgowania"], dayfirst=True)
df["Miesiąc"] = df["Data księgowania"].dt.to_period("M")
df["Miesiąc_etykieta"] = df["Miesiąc"].dt.strftime("%b %Y")
df = df.sort_values("Miesiąc")

In [8]:
df["Kwota operacji"] = (
    df["Kwota operacji"]
    .astype(str)
    .str.replace(" ", "", regex=False)   # remove spaces
    .str.replace(",", ".", regex=False)  # handle comma as decimal if needed
    .astype(float)
) * (-1)
df = df[df["Kwota operacji"] > 0]

In [ ]:
monthly = (
    df.groupby(["Miesiąc", "Miesiąc_etykieta", "Kategoria_2"])["Kwota operacji"]
    .sum()
    .reset_index()
)

monthly_totals = (
    df.groupby(["Miesiąc", "Miesiąc_etykieta"])["Kwota operacji"]
    .sum()
    .reset_index()
    .rename(columns={"Kwota operacji":"Razem"})
)

monthly = monthly.merge(monthly_totals)
monthly["Procent"] = monthly["Kwota operacji"] / monthly["Razem"] * 100
monthly = monthly.sort_values(["Miesiąc", "Kwota operacji"], ascending=[True, False])

cat_month = (
    df.groupby(["Miesiąc", "Miesiąc_etykieta", "Kategoria_2", "Kategoria"])["Kwota operacji"]
    .sum()
    .reset_index()
)

In [ ]:
monthly

In [ ]:
cat_month

In [ ]:
fig = px.bar(
    monthly,
    x="Miesiąc_etykieta",
    y="Kwota operacji",
    color="Kategoria_2",
    title="Miesięczne wydatki na Nadkategorię (per-month largest at bottom)"
)

fig.update_layout(
    xaxis_title="Miesiąc",
    yaxis_title="Kwota",
    legend_title="Kategoria_2",
    barmode="stack",
)

fig.show()

In [ ]:
fig = px.bar(
    cat_month,
    x="Miesiąc_etykieta",
    y="Kwota operacji",
    color="Kategoria",
    facet_row="Kategoria_2",
    title="Miesięczne wydatki na Kategorię",
    facet_row_spacing=0.04
)

fig.update_yaxes(matches=None)

for i, yaxis in enumerate(fig.layout, start=1):
    if yaxis.startswith("yaxis"):
        fig.layout[yaxis].title.text = ""

for ann in fig.layout.annotations:
    if "Kategoria_2=" in ann.text:
        ann.text = ann.text.split("=")[1]
        
        ann.y += 0.03
        
        ann.x = 0
        ann.xref = "paper"
        
        ann.textangle = 0
        ann.font.size = 12
        ann.align = "center"
        
for i, xaxis in enumerate(fig.layout, start=1):
    if xaxis.startswith("xaxis"):
        fig.layout[xaxis].showticklabels = True
        fig.layout[xaxis].title.text = ""

fig.update_layout(height=2400)
fig.show()

In [ ]:
fig = px.sunburst(
    df,
    path=["Miesiąc_etykieta", "Kategoria_2", "Kategoria"],
    values="Kwota operacji",
    title="Hierarchia wydatków"
)
fig.update_layout(
    height=1500,  # bigger
    width=1500
)
fig.show()